In [79]:
from tf_ofcs.models import IMDDAutoencoder
from tf_ofcs.__main__ import _load_weights
from tf_ofcs.utils import index_to_gray, gray_to_binary
import itertools

import tensorflow as tf

In [7]:
ae = IMDDAutoencoder(M=64, n=48, rx_config={"N": 15, "n_layers": 15}, tx_config={"N": 25, "n_layers": 3})
_load_weights(ae.model, "../docker/output/imdd/imdd-M64-n48-N_tx25-N_rx15-TX3-RX15-0-60km_0-20ps.keras")

In [21]:
table = []

for bitseq in itertools.product((0, 1), repeat=6):
    table.append(ae.encoder([bitseq]).samples[:, 48:-48])

table = tf.concat(table, 0)

In [62]:
mse_table = tf.vectorized_map(lambda i: tf.math.reduce_mean((table[i] - table) ** 2, axis=-1), tf.range(64))
mse_table += tf.math.multiply_no_nan(tf.eye(64, 64, dtype=tf.float32), 1e32)

In [63]:
best_matches = tf.argmin(mse_table, axis=-1)

In [64]:
best_matches

<tf.Tensor: shape=(64,), dtype=int64, numpy=
array([14, 21, 28, 55, 55, 55,  4,  1,  4,  5,  7, 35, 28,  8, 32, 41, 34,
       27, 31, 60, 22,  1, 56, 43, 47, 63, 59, 17, 33, 58, 44, 34, 14, 52,
       60, 11, 52, 51, 14, 24, 55, 16, 56, 60, 30, 20, 19, 60, 56, 44, 16,
       37, 33, 35, 44,  4, 22, 31, 14, 21, 34,  0, 37, 25])>

In [68]:
mapping = []
already_picked = set()
for i in range(64):
    best_candidates = sorted(list(range(64)), key=lambda j: mse_table[i][j])

    best_candidate = None
    
    for j in best_candidates:
        if j not in already_picked:
            best_candidate = j
            already_picked.add(j)
            break

    assert best_candidate is not None
    mapping.append(best_candidate)

In [71]:
repr(mapping)

'[14, 21, 28, 55, 32, 9, 4, 1, 13, 5, 7, 35, 33, 8, 58, 41, 34, 27, 31, 60, 22, 59, 56, 43, 47, 63, 51, 17, 2, 6, 44, 57, 38, 52, 19, 11, 54, 62, 40, 24, 61, 16, 20, 23, 30, 25, 49, 3, 29, 37, 53, 0, 12, 50, 36, 10, 48, 15, 42, 26, 18, 45, 46, 39]'

In [96]:
new_mapping = list(range(64))

for i in range(64):
    new_mapping[int(index_to_gray(i))] = mapping[i]

In [97]:
repr(new_mapping)

'[14, 21, 55, 28, 1, 4, 32, 9, 41, 58, 33, 8, 13, 5, 35, 7, 57, 44, 2, 6, 47, 63, 17, 51, 34, 27, 60, 31, 43, 56, 22, 59, 39, 46, 18, 45, 48, 15, 26, 42, 29, 37, 0, 53, 10, 36, 12, 50, 38, 52, 11, 19, 24, 40, 54, 62, 3, 49, 30, 25, 61, 16, 23, 20]'